# 05 Pymefpp Scamp

This notebook run SExtractor and Scamp over the mosaic images resulting from WCS solution retrieves by solve-field in notebook 4.

The flow is as follows:
- For each image with a wcs mosaic will first run SExtractor to create a FITS_LDAC catalogue of the stars in the field.
- Once the catalogue is created, Scamp is called. It will first extract the headers of the mosaic to create a .ahead file which will additionaly include airmass and filter header. Scamp will be called over the FITS_LDAC catalogue creating a .head file with the improved WCS headers.

Tested on my laptop, takes approximately 1 minute by mosaic. <br>
Tested with SExtractor 2.25 and Scamp 2.0.4. Please consider that older versions of SExtractor are called with /usr/bin/sex, but the newest (the one used here) are called with /usr/bin/source-extractor. I remember that in Chapman SExtractor must be called in the old way, so you will have to modified the SExtractor.run method to call the program correctly. The differences also includes the SExtractor parameters. Some parameters from the original file (for the older version) rises troubles when included in the new version. I have not tested if the 2.25 configuration works well on older versions.
If needed, I also include the old parameter configuration.

All the functions that relates with SExtractor or Scamp, like the creation of the configuration files, and the execution of the programs are inside their respective class within the file astromatic.py

In [ ]:
import os
import glob
import logging
import sys
from mefpp_astromatic import SExtractor, Scamp

In [ ]:
def prepare_sextractor(filename_abspath:str, log_dir:str, redo_sextractor:bool = False):
    """
    Prepare the directories necessaries to run Sextractor prior its execution.
    It checks if the output files are present. If it is the case, then skipped the respective execution.
    
    Parameters
    ----------
    filename_abspath: str
        The absolute path to the wcs mosaic.
        
    log_dir: str
        the path to log folder.
    
    redo_sextractor: bool
        If True will rerun SExtractor even if the .cat fits file exists.
    """
    
    # define the paths to the sextractor and scamp folders and created it
    dir_name = "/".join(filename_abspath.split("/")[:-1])
    sextractor_dir_path = "/".join(filename_abspath.split("/")[:-4])+"/Sextractor_dir"
    
    log_file_sextractor = log_dir + "/pymeffpp_sextractor.log"
    
    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s:%(levelname)s:%(message)s",
                        handlers=[
                                  logging.FileHandler(log_file_sextractor),
                                  logging.StreamHandler()])
    logging.getLogger().setLevel(logging.INFO)
    
    # creates the Sextractor folder where the log files will be located
    try:
        logging.info("Atempting to create the Sextractor folder")
        os.mkdir(sextractor_dir_path)
        logging.info("Folder created")
    except:
        logging.info("Folder already exists")

    return dir_name, sextractor_dir_path, log_file_sextractor

def run_sextractor_on_target(filename_abspath, sextractor_dir_path):
    """
    Run sextractor over the wcs.fits image created by solve_field.
    The final output is a catalogue of stars in the mosaic in format FITS_LDAC, ready to be read by Scamp.
    """

    log_file_sextractor = sextractor_dir_path + "/pymeffpp_sextractor.log"
    
    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s:%(levelname)s:%(message)s",
                        handlers=[
                                  logging.FileHandler(log_file_sextractor),
                                  logging.StreamHandler()])
    logging.getLogger().setLevel(logging.INFO)

    # Due to limitations in path length of astromatic we need to work 
    # in the directory where the files to be sextracted are located.
    wcs_image_path =  '/'.join(filename_abspath.split("/")[:-1])
    logging.info("Moving to target's directory: {}".format(wcs_image_path))
    current_dir = os.getcwd()
    os.chdir(wcs_image_path)
    
    # we call the Sextractor class which will created all the files necessaries to run sextractor
    # then just run it
    # that will create the output necessary for Scamp
    
    sextractor = SExtractor(filename_abspath, sextractor_dir_path)
    p = sextractor.run_sextractor()
    if p == 1:
        logging.info("Sextraction failed")
    elif p == 0:
        logging.info("Sextraction finished without problems")
        
    os.chdir(current_dir)
    return sextractor

def prepare_scamp(filename_abspath:str, log_dir:str, redo_scamp:bool = False):
    """
    Prepare the directories necessaries to run Sextractor and Scamp prior their execution.
    It checks if the output files are present. If it is the case, then skipped the respective execution.
    
    Parameters
    ----------
    filename_abspath: str
        The absolute path to the wcs mosaic.
        
    log_dir: str
        the path to log folder.

    redo_scamp: bool
        If True will rerun Scamp even if the .head file exists.
        
    """
    
    # define the paths to the sextractor and scamp folders and created it
    dir_name = "/".join(filename_abspath.split("/")[:-1])
    scamp_dir_path = "/".join(filename_abspath.split("/")[:-4])+"/Scamp_dir"
    log_file_scamp = log_dir + "/pymeffpp_scamp.log"
    
    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s:%(levelname)s:%(message)s",
                        handlers=[
                                  logging.FileHandler(log_file_scamp),
                                  logging.StreamHandler()])
    logging.getLogger().setLevel(logging.INFO)
    
    # creates the Scamp folder where the log files will be located
    try:
        logging.info("Atempting to create the Scamp folder")
        os.mkdir(scamp_dir_path)
        logging.info("Folder created")
    except:
        logging.info("Folder already exists")

    # now the folder are created, it is moment to run sextractor first
    # first move the folder where the data are
    logging.info("Moving to target's directory: {}".format(dir_name))
    current_dir = os.getcwd()
    os.chdir(dir_name)
    # check if the sextractor catalogue exists, if it does then continue, unless the flag "redo_sextractor"
    # is True
    # run sextractor if it is the case
    

    
    # the process is done, we can return to the original folder
    logging.info("SExtractor + Scamp finished on image {}".format(filename_abspath.split("/")[-1]))
    logging.info("Returning to original directory: {}".format(current_dir))
    os.chdir(current_dir)


def run_scamp_on_target(filename_abspath, scamp_dir_path):
    """
    Run Scamp over the wcs.fits image.
    It creates a .ahead file from the image header, and uses the catalogue output by Sextractor.
    As final output a .head file with the corrected WCS headers is created, which will be later read by Swarp.
    """

        # Due to limitations in path length of astromatic we need to work 
    # in the directory where the files to be sextracted are located.
    wcs_image_path =  '/'.join(filename_abspath.split("/")[:-1])
    logging.info("Moving to target's directory: {}".format(wcs_image_path))
    current_dir = os.getcwd()
    os.chdir(wcs_image_path)
    
    scamp = Scamp(filename_abspath, scamp_dir_path)  # Initialize the Scamp class with default configuration
    
    # change some parameters from default
    scamp.config["ASTREF_BAND"] = "REDDEST"
    scamp.config["ASTREFMAG_LIMITS"] = [10, 18]
    scamp.config["POSITION_MAXERR"] = 2
    scamp.config["MOSAIC_TYPE"] = "FIX_FOCALPLANE"
    scamp.config["CROSSID_RADIUS"] = 5.0
    scamp.config["SN_THRESHOLDS"] = [10.0, 40.0]
    scamp.config["FWHM_THRESHOLDS"] = [1.0, 10.0]
    scamp.config["POSANGLE_MAXERR"] = 2.5
    
    scamp.write_ahead_file()  # creates the .ahead file with the headers of the image
    scamp.run(scamp.cat, updateconfig=True, clean=False)  # run scamp on the single image
    
    logging.info("Scamp finished without problems. Check the diagnosis plots for more information.")

    os.chdir(current_dir)

    return scamp

In [ ]:
# set the directory folders
# same as the rest of notebooks
#root_dir = "/Users/fselman/Pictures/OCAM/CPPup/reduced/"
root_dir = "/Users/fselman/Pictures/OCAM/M67/"
reduced_science_dir = root_dir+'science/'
log_dir = root_dir+'logs/'

# here we select all the folders with the _pre_wcs.fits mosaic 

targets_to_scamp = glob.glob(reduced_science_dir+"/*/*/*/*_pre/*_wcs.fits")
targets_to_scamp.sort()

c = 0 
for target in targets_to_scamp:
    c += 1
    
    # creates the Scamp folder where the log files will be located
    scamp_dir_path = "/".join(target.split("/")[:-4])+"/Scamp_dir/"
    
    try:
        logging.info("Atempting to create the Scamp_dir folder")
        os.mkdir(scamp_dir_path)
        logging.info("Folder created")
    except:
        logging.info("Folder already exists")

    log_file_scamp_sextractor = scamp_dir_path + "/pymeffpp_scamp_sextractor.log"
    
    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s:%(levelname)s:%(message)s",
                        handlers=[
                                  logging.FileHandler(log_file_scamp_sextractor),
                                  logging.StreamHandler()])
    logging.getLogger().setLevel(logging.INFO)

    logging.info("--------------------------------------------------------------------------------------")
    logging.info("Preparing to Sextract and Scamp fits image : {:d}/{:d}".format(c, len(targets_to_scamp)))
    logging.info("Fits image : {}".format(target.split("/")[-1]))
    
    wcs_image_path =  '/'.join(target.split("/")[:-1])

    # here we count backward as the project directory could be any
    target_name = target.split('/')[-5]
    filter_name = target.split('/')[-4]
    logging.info("             _target: {}".format(target))
    logging.info("               path: {}".format('reduced_science_dir + target_dir + /' + '/'.join(target.split("/")[9:-1])))
    logging.info("             target: {}".format(target_name))
    logging.info("             filter: {}".format(filter_name))
    
    # SOURCE-EXTRACTOR ON OUTPUT OF SOLVE-FIELD
    #
    # we create the folders needed by sextractor
    # now the folder are created, it is moment to run sextractor first
    # first move the folder where the data are. This changing of directories
    # is needed because sorce-extractor cannot handle very long paths in its
    # input file
    #
    sextractor_dir_path = reduced_science_dir+target_name+'/Sextractor_dir/'
    try:
        logging.info("Atempting to create the Sextractor_dir folder: {}".format(reduced_science_dir+target_name))
        os.mkdir(sextractor_dir_path)
        logging.info("Folder created")
    except:
        logging.info("Folder already exists")
        
    # This flag can safely left to False unless you decide otherwise.
    # It can save some time.
    #
    redo_sextractor = True
    sextractor_cat_file = target[:-5] + ".cat"
    if not os.path.isfile(sextractor_cat_file):
        logging.info("Running Sextractor on image {}".format(target.split("/")[-1]))
        run_sextractor_on_target(target, sextractor_dir_path)
    elif os.path.isfile(sextractor_cat_file) and redo_sextractor:
        logging.info("Removing {} before proceeding".format(sextractor_cat_file))
        os.remove(sextractor_cat_file)
        os.sync()
        run_sextractor_on_target(target, sextractor_dir_path)
    else:
        logging.info("Catalogue {} already exists and redo is False. Skipping Sextractor".format(sextractor_cat_file))
    # sextractor is done
    logging.info("SExtractor finished on image {}".format(target.split("/")[-1]))

    # SCAMP ON OUTPUT OF SOLVE WITH THE SOURCE_EXTRACTOR CATALOGUE
    #
    # At this point Sextractor is finished and we can start to run Scamp
    # check if the Scamp .head file exists, same procedure as for Sextractor
    scamp_dir_path = reduced_science_dir+target_name+'/Scamp_dir/'
    try:
        logging.info("Atempting to create the Scamp_dir folder: {}".format(scamp_dir_path))
        os.mkdir(scamp_dir_path)
        logging.info("Folder created")
    except:
        logging.info("Folder already exists")
    scamp_head_file = target[:-5]+".head"
    
    # At this point Sextractor is finished and we can start to run Scamp
    # check if the Scamp .head file exists, same procedure as for Sextractor

    # This flag must be kept True because the script fails if it finds 
    # the file from a previous run.
    redo_scamp = True
    if os.path.isfile(sextractor_cat_file):
        if not os.path.isfile(scamp_head_file):
            logging.info("Running Scamp on image {}".format(target.split("/")[-1]))
            if os.path.isfile(scamp_head_file[:-5]+".ahead"):
                os.remove(scamp_head_file[:-5]+".ahead")
                os.sync()
            run_scamp_on_target(target, scamp_dir_path)
        elif os.path.isfile(scamp_head_file) and redo_scamp:
            logging.info("Removing {} before proceeding".format(scamp_head_file))
            os.remove(scamp_head_file)
            os.sync()
            if os.path.isfile(scamp_head_file[:-5]+".ahead"):
                os.remove(scamp_head_file[:-5]+".ahead")
                os.sync()
            run_scamp_on_target(target, scamp_dir_path)
        else:
            logging.info("Header file {} already exists. Skipping Scamp".format(scamp_head_file))
    else:
            logging.info("Cannot find source-extractor catalogue: {}".format(sextractor_file))

    
logging.info('-------------- DONE ---------------')  

In [ ]:
# Old sextractor configuration, If needed to change, please go to the SExtractor.write_default_param function in
# mefpp_astromatic.py file, and replace the lines by these ones (line 121 within code).

lines = """NUMBER                   Running object number                                    
EXT_NUMBER               FITS extension number                                    
FLUX_ISO               #  Isophotal flux                                            [count]
FLUXERR_ISO            #  RMS error for isophotal flux                              [count]
MAG_ISO                #  Isophotal magnitude                                       [mag]
MAGERR_ISO             #  RMS error for isophotal magnitude                         [mag]
FLUX_APER              #  Flux vector within fixed circular aperture(s)             [count]
FLUXERR_APER           #  RMS error vector for aperture flux(es)                    [count]
MAG_APER               #  Fixed aperture magnitude vector                           [mag]
MAGERR_APER            #  RMS error vector for fixed aperture mag.                  [mag]
FLUX_AUTO              ##  Flux within a Kron-like elliptical aperture               [count]
FLUXERR_AUTO           #  RMS error for AUTO flux                                   [count]
MAG_AUTO               #  Kron-like elliptical aperture magnitude                   [mag]
MAGERR_AUTO            #  RMS error for AUTO magnitude                              [mag]
A_IMAGE                  Profile RMS along major axis                              [pixel]
B_IMAGE                  Profile RMS along minor axis                              [pixel]
THETA_IMAGE              Position angle (CCW/x)                                    [deg]
XWIN_IMAGE               Windowed position estimate along x                        [pixel]
YWIN_IMAGE               Windowed position estimate along y                        [pixel]
ERRX2WIN_IMAGE           Variance of windowed pos along x                          [pixel**2]
ERRY2WIN_IMAGE           Variance of windowed pos along y                          [pixel**2]
ERRXYWIN_IMAGE           Covariance of windowed pos between x and y                [pixel**2]
ERRX2WIN_WORLD           Variance of windowed pos along X-WORLD (alpha)            [deg**2]
ERRY2WIN_WORLD           Variance of windowed pos along Y-WORLD (delta)            [deg**2]
ERRXYWIN_WORLD           Covariance of windowed pos X-WORLD/Y-WORLD                [deg**2]
ERRAWIN_IMAGE            RMS windowed pos error along major axis                   [pixel]
ERRBWIN_IMAGE            RMS windowed pos error along minor axis                   [pixel]
ERRTHETAWIN_IMAGE        Windowed error ellipse pos angle (CCW/x)                  [deg]
FLAGS                    Extraction flags                                         
FLAGS_WEIGHT             Weighted extraction flags                                
FLAGS_WIN                Flags for WINdowed parameters                            
FWHM_IMAGE               FWHM assuming a gaussian core                             [pixel]
ELONGATION               A_IMAGE/B_IMAGE                                          
ELLIPTICITY              1 - B_IMAGE/A_IMAGE                                      
CLASS_STAR                S/G classifier output                                    
FLUX_RADIUS            #  Fraction-of-light radii                                   [pixel]"""